# 02 · Banana Classifier Workbench
Classify images, run batch jobs, probe confidence scores, and audit decision history.

**Covered endpoints**
- `POST /banana` — classify a single item
- `GET  /banana/summary` — aggregate stats
- `POST /batch` — queue a batch job
- `GET  /batch/{id}` — poll batch status
- `GET  /not-banana/summary` — non-banana classification summary
- `POST /not-banana` — classify a non-banana sample

In [ ]:
# ── Shared setup (copy from 01-platform-setup if running standalone) ───────
import json, urllib.request, urllib.error
API_BASE = 'https://api.banana.engineer'

def pp(obj):
    if isinstance(obj, str):
        try: obj = json.loads(obj)
        except Exception: print(obj); return
    print(json.dumps(obj, indent=2))

def call_endpoint(method, path, payload=None, headers=None):
    url = f"{API_BASE}{path}"
    h = {'Accept': 'application/json'}
    if headers: h.update(headers)
    data = None
    if payload is not None:
        h['Content-Type'] = 'application/json'
        data = json.dumps(payload).encode()
    req = urllib.request.Request(url, data=data, headers=h, method=method.upper())
    try:
        with urllib.request.urlopen(req) as r:
            return {'ok': True, 'status': r.status, 'body': r.read().decode('utf-8', errors='replace')}
    except urllib.error.HTTPError as e:
        return {'ok': False, 'status': e.code, 'body': e.read().decode('utf-8', errors='replace')}
    except Exception as ex:
        return {'ok': False, 'status': None, 'body': str(ex)}

def get(path, **kw):  return call_endpoint('GET',  path, **kw)
def post(path, **kw): return call_endpoint('POST', path, **kw)
print('setup_ok')

In [ ]:
# ── 1. Banana summary ─────────────────────────────────────────────────────
r = get('/banana/summary')
print('status:', r['status'])
pp(r['body'])

In [ ]:
# ── 2. Single classification ──────────────────────────────────────────────
# Adjust payload fields to match your current BananaRequest schema
payload = {
    'purchases': 3,
    'multiplier': 2
}
r = post('/banana', payload=payload)
print('status:', r['status'])
pp(r['body'])

In [ ]:
# ── 3. Batch classification probe ────────────────────────────────────────
samples = [
    {'purchases': 1, 'multiplier': 2},
    {'purchases': 5, 'multiplier': 3},
    {'purchases': 2, 'multiplier': 1},
]
results = []
for s in samples:
    r = post('/banana', payload=s)
    try:
        body = json.loads(r['body'])
    except Exception:
        body = r['body']
    results.append({'input': s, 'status': r['status'], 'output': body})

print(f'batch_complete | samples: {len(results)}')
for row in results:
    print(row)

In [ ]:
# ── 4. Not-banana summary ────────────────────────────────────────────────
r = get('/not-banana/summary')
print('status:', r['status'])
pp(r['body'])

In [ ]:
# ── 5. Confidence spread analysis ────────────────────────────────────────
# Sweep multiplier values and measure banana score spread
sweep = [{'purchases': 3, 'multiplier': m} for m in range(1, 8)]
scores = []
for s in sweep:
    r = post('/banana', payload=s)
    if r['ok']:
        try:
            body = json.loads(r['body'])
            scores.append({'multiplier': s['multiplier'], 'result': body})
        except Exception:
            pass

print(f'sweep_done | {len(scores)} results')
for row in scores:
    print(row)